In [141]:
from minio import Minio
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from pyspark.sql.types import NumericType, IntegerType, LongType, FloatType, DoubleType, DecimalType, DateType, TimestampType
from pyspark.sql import Window
import pyspark.sql.functions as F
from dotenv import load_dotenv, find_dotenv
import psycopg2
load_dotenv(find_dotenv())
import os

JAR_PATH_1 = os.path.abspath("./jars/hadoop-aws-3.4.0.jar")
JAR_PATH_2 = os.path.abspath("./jars/aws-sdk-s3-2.29.52.jar")

JARS_LIST = f"{JAR_PATH_1},{JAR_PATH_2}"

spark = (
    SparkSession.builder.appName("analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.jars.repositories", "https://repo1.maven.org/maven2/")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

In [142]:
spark = (
    SparkSession.builder.appName("analysis")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

In [143]:
spark

In [144]:
conn = psycopg2.connect(
    host="localhost",
    database=os.getenv("POSTGRES_DATABASE_NAME"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)


LOAD DATA FROM POSTGRE DATABASE REMAINING

In [145]:
import os
import psycopg2
from pyspark.sql.types import StructType, StructField, StringType


DB_HOST = "localhost" 
DB_PORT = "5432" 
DB_NAME = os.getenv("POSTGRES_DATABASE_NAME", "pulse")
DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASS = os.getenv("POSTGRES_PASSWORD", "postgres")

def get_agg_tables():
    try:
        print(f"Connecting to Postgres at {DB_HOST}:{DB_PORT}...")
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASS
        )
        cursor = conn.cursor()

        cursor.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' 
            AND table_name LIKE 'agg_%'
        """)
        
        tables = [row[0] for row in cursor.fetchall()]
        print(f"Found tables: {tables}")

        spark_dfs = {}

        jdbc_url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
        connection_properties = {
            "user": DB_USER,
            "password": DB_PASS,
            "driver": "org.postgresql.Driver"
        }

        for table in tables:
            print(f"Processing table: {table}...")
            df = spark.read.jdbc(url=jdbc_url, table=f'"{table}"', properties=connection_properties)
            spark_dfs[table]= df

        cursor.close()
        conn.close()
        return spark_dfs

    except Exception as e:
        print(f"Error: {e}")
        # Print full stack trace for debugging if needed
        import traceback
        traceback.print_exc()
        return {}

dataframes = get_agg_tables()

Connecting to Postgres at localhost:5432...
Found tables: ['agg_customer_sessions', 'agg_customers', 'agg_inventory', 'agg_marketing_campaigns', 'agg_order_items', 'agg_orders', 'agg_payments', 'agg_products', 'agg_reviews', 'agg_shopping_cart', 'agg_suppliers', 'agg_wishlist', 'agg_categories', 'agg_daily_aggregations', 'agg_weekly_aggregations', 'agg_monthly_aggregations', 'agg_country_aggregations', 'agg_state_aggregations', 'agg_city_aggregations', 'agg_cart_abandonment_analysis', 'agg_product_inventory_health', 'agg_supplier_inventory_health', 'agg_rfm_segmentation', 'agg_rfm_segment_summary', 'agg_product_affinity', 'agg_top_product_pairs', 'agg_product_recommendations', 'agg_category_affinity', 'agg_global_aggregations']
Processing table: agg_customer_sessions...
Processing table: agg_customers...
Processing table: agg_inventory...
Processing table: agg_marketing_campaigns...
Processing table: agg_order_items...
Processing table: agg_orders...
Processing table: agg_payments...
P

In [146]:
dataframes["agg_rfm_segment_summary"].show(5)

+----------------------+--------------+-----------+----------+--------------------+-------------+
|customer_segment_label|customer_count|avg_revenue|avg_orders|avg_days_since_order|avg_rfm_score|
+----------------------+--------------+-----------+----------+--------------------+-------------+
+----------------------+--------------+-----------+----------+--------------------+-------------+



ALL NULL ROWs AND DATAFRAME CHECK 

In [147]:
def is_column_all_null_or_zero(df, col_name):
    if df is None:
        return True                    

    if col_name not in df.columns:
        return True                   

    col_type = dict(df.dtypes)[col_name]
    non_null_count = df.agg(
        F.count(F.col(col_name)).alias("non_null_count")
    ).collect()[0]["non_null_count"]
    if non_null_count == 0:
        return True                   

    
    if col_type in ("int", "bigint", "double", "float", "decimal", "smallint", "tinyint"):
        non_zero_non_null_count = df.agg(
            F.sum(
                F.when(
                    (F.col(col_name).isNotNull()) & (F.col(col_name) != 0), 1
                ).otherwise(0)
            ).alias("non_zero_non_null_count")
        ).collect()[0]["non_zero_non_null_count"]

        if non_zero_non_null_count == 0:
            return True                 

    return False

Time Grain function

In [148]:
def add_time_grain(df, date_col, grain="day"):
    if grain == "day":
        return df.withColumn("grain_date", F.col(date_col))
    elif grain == "week":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_week", F.weekofyear(date_col))
    elif grain == "month":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_month", F.month(date_col))
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

# Customer Related Analysis 

Adding Date Column

In [153]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    dataframes["agg_customers"] = dataframes["agg_customers"].withColumn(
    "account_created_date",
    F.to_date("account_created_at")
    )

Analysis: Net Revenue vs. Net Profit financial health over time 

In [154]:
def analyze_financial_health(orders_df, date_col="order_placed_at", grain="month"):
    df_g = add_time_grain(orders_df, date_col, grain)
    
    if grain == "day":
        group_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
    else: # month
        group_cols = ["grain_year", "grain_month"]

    financial_df = (
        df_g.groupBy(*group_cols)
            .agg(
                F.sum("net_revenue").alias("total_net_revenue"),
                F.sum("net_profit").alias("total_net_profit"),
                F.count("order_id").alias("total_orders")
            )
            .withColumn("period_margin_pct", 
                        F.round((F.col("total_net_profit") / F.col("total_net_revenue")) * 100, 2))
            .orderBy(*group_cols)
    )
    
    return financial_df
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_placed_at"):
    daily_health = analyze_financial_health(dataframes["agg_orders"],"order_placed_at", grain="day")
    monthly_health = analyze_financial_health(dataframes["agg_orders"], "order_placed_at", grain="month")

In [155]:
monthly_health.show(3)

+----------+-----------+-----------------+-------------------+------------+-----------------+
|grain_year|grain_month|total_net_revenue|   total_net_profit|total_orders|period_margin_pct|
+----------+-----------+-----------------+-------------------+------------+-----------------+
|      2023|         11|          5647.75|-261524.88999999998|          32|          -4630.6|
|      2023|         12|          17408.3| -611248.9100000001|          85|         -3511.25|
|      2024|          1|         18754.68|-481553.85000000015|          75|         -2567.65|
+----------+-----------+-----------------+-------------------+------------+-----------------+
only showing top 3 rows



Analysis: Margin by Category (The "Drag" Analysis)

In [156]:
def analyze_category_margins(products_df):
   
    category_df = (
        products_df.groupBy("category")
            .agg(
                F.avg("profit_margin").alias("avg_profit_margin"),
                F.sum("total_profit").alias("total_category_profit"),
                F.sum("total_revenue").alias("total_category_revenue"),
                F.sum("total_units_sold").alias("units_sold")
            )
            .orderBy(F.col("avg_profit_margin").asc())
    )
    
    return category_df

if not is_column_all_null_or_zero(dataframes["agg_products"], "category"):
    low_margin_cats = analyze_category_margins(dataframes["agg_products"])
else: 
    print("Category column is all NULL or zero; skipping category margin analysis.")

active customers over time

In [157]:

def active_customers_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.filter(F.col("is_active") == True)
          .groupBy(*group_cols)
          .agg(F.countDistinct("customer_id").alias("active_customers"))
          .orderBy(*group_cols)
    )
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    daily_active   = active_customers_over_time(dataframes["agg_customers"], "day")
    weekly_active  = active_customers_over_time(dataframes["agg_customers"], "week")
    monthly_active = active_customers_over_time(dataframes["agg_customers"], "month")
else:
    print("Account created at column is all NULL or zero; skipping active customers over time analysis.")

account_status over time 

In [158]:
def status_distribution_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.groupBy(*(group_cols + [F.col("account_status")]))
          .agg(F.countDistinct("customer_id").alias("customer_count"))
          .orderBy(*group_cols, "account_status")
    )
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    daily_status   = status_distribution_over_time(dataframes["agg_customers"], "day")
    monthly_status = status_distribution_over_time(dataframes["agg_customers"], "month")
else:
    print("Account created at column is all NULL or zero; skipping status distribution over time analysis.")

New customers per day/week/month

In [159]:
def new_customers(df, date_col="account_created_at", grain="day"):
    df_g = add_time_grain(df, date_col=date_col, grain=grain)

    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    else:   # month
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]

    new_df = (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .orderBy(*order_cols)
    )
    return new_df
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    daily_new   = new_customers(dataframes["agg_customers"],"account_created_at", "day")
    weekly_new  = new_customers(dataframes["agg_customers"], "account_created_at", "week")
    monthly_new = new_customers(dataframes["agg_customers"], "account_created_at", "month")
else:
    print("Account created at column is all NULL or zero; skipping new customers analysis.")

Cumulative customer growth curve

In [160]:
def cumulative_customers(df, date_col="account_created_at", grain="day"):
    new_df = new_customers(df, date_col, grain)

    # Define window by time order
    if grain == "day":
        window = Window.orderBy("grain_date") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    elif grain == "week":
        window = Window.orderBy("grain_year", "grain_week") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    else:   # month
        window = Window.orderBy("grain_year", "grain_month") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    cum_df = new_df.withColumn(
        "cumulative_customers",
        F.sum("new_customers").over(window)
    )
    return cum_df
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    daily_growth   = cumulative_customers(dataframes["agg_customers"],"account_created_at", "day")
    weekly_growth  = cumulative_customers(dataframes["agg_customers"], "account_created_at", "week")
    monthly_growth = cumulative_customers(dataframes["agg_customers"], "account_created_at", "month")
else:
    print("Account created at column is all NULL or zero; skipping cumulative customers analysis.")

Total new customers by geography + time

In [161]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    geo_acquisition = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "city")
        .agg(F.countDistinct("customer_id").alias("new_customers"))
    )
else:
    print("Account created at column is all NULL or zero; skipping geo acquisition analysis.")

def geo_acquisition_over_time(df, date_col="account_created_at", grain="day"):
    df_g = add_time_grain(df, date_col=date_col, grain=grain)

    if grain == "day":
        group_cols = ["grain_date", "country", "state_province", "city"]
        order_cols = ["grain_date", "country", "state_province", "city"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
    else:  # month
        group_cols = ["grain_year", "grain_month", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_month", "country", "state_province", "city"]

    return (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .orderBy(*order_cols)
    )

if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    daily_geo   = geo_acquisition_over_time(dataframes["agg_customers"], "account_created_at", "day")
    monthly_geo = geo_acquisition_over_time(dataframes["agg_customers"], "account_created_at", "month")
else:
    print("Account created at column is all NULL or zero; skipping geo acquisition over time analysis.")

Customer distribution by age group, city, state, country

In [162]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_age_group"):
    age_group_dist = (
        dataframes["agg_customers"]
        .groupBy("customer_age_group")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("customer_age_group")
    )
else: 
    print("Customer age group column is all NULL or zero; skipping age group distribution analysis.")
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province") and not is_column_all_null_or_zero(dataframes["agg_customers"], "city"):
    city_dist = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "city")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("country", "state_province", "city")
)
else:
    print("Country column is all NULL or zero; skipping city distribution analysis.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province"):
    state_dist = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("country", "state_province")
)
    
else:
    print("Country or state_province column is all NULL or zero; skipping state distribution analysis.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "country"):
    country_dist = (
        dataframes["agg_customers"]
        .groupBy("country")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country")
)
else:
    print("Country column is all NULL or zero; skipping country distribution analysis.")

# Age group distribution and spending patterns

In [163]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_age_group") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_total_spent") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue"):
    age_group_spending = (
        dataframes["agg_customers"]
        .groupBy("customer_age_group")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("order_total_spent").alias("avg_order_total_spent"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.sum("order_total_spent").alias("total_spent"),
            F.sum("total_revenue").alias("total_revenue_age_group")
        )
        .orderBy("customer_age_group")
    )
else: 
    print("Customer age group column is all NULL or zero; skipping age group spending analysis.")

Gender-based product preferences

In [164]:
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_orders"], "customer_id"):
    cust_orders = (
        dataframes["agg_orders"]
        .select("order_id", "customer_id")
        .join(
            dataframes["agg_customers"].select("customer_id", "gender"),
            on="customer_id",
            how="inner"
        )
    )
else:
    print("Order ID or Customer ID column is all NULL or zero; skipping customer orders join.")

if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_order_items"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_order_items"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_order_items"], "quantity") and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name") and not is_column_all_null_or_zero(dataframes["agg_products"], "category") and not is_column_all_null_or_zero(dataframes["agg_products"], "sub_category") and not is_column_all_null_or_zero(dataframes["agg_products"], "brand"):
    cust_order_items = (
        cust_orders
        .join(dataframes["agg_order_items"].select("order_id", "product_id", "quantity"), on="order_id", how="inner")
        .join(dataframes["agg_products"].select("product_id", "product_name", "category", "sub_category", "brand"),
            on="product_id",
            how="left")
    )
else:
    print("One of the required columns in orders, order items, or products is all NULL or zero; skipping customer order items join.")
# Gender-based preferences by category
if not is_column_all_null_or_zero(cust_order_items, "gender") and not is_column_all_null_or_zero(cust_order_items, "category") and not is_column_all_null_or_zero(cust_order_items, "quantity") and not is_column_all_null_or_zero(cust_order_items, "product_id") and not is_column_all_null_or_zero(cust_order_items, "order_id"):    
    gender_category_pref = (
        cust_order_items
        .groupBy("gender", "category")
        .agg(
            F.sum("quantity").alias("total_units"),
            F.countDistinct("product_id").alias("distinct_products"),
            F.countDistinct("order_id").alias("orders_count")
        )
        .orderBy("gender", F.col("total_units").desc())
    )
else:
    print("One of the required columns in customer order items is all NULL or zero; skipping gender category preference analysis.")

if not is_column_all_null_or_zero(cust_order_items, "gender") and not is_column_all_null_or_zero(cust_order_items, "product_id") and not is_column_all_null_or_zero(cust_order_items, "product_name") and not is_column_all_null_or_zero(cust_order_items, "category") and not is_column_all_null_or_zero(cust_order_items, "quantity") and not is_column_all_null_or_zero(cust_order_items, "order_id"):
    gender_product_pref = (
        cust_order_items
        .groupBy("gender", "product_id", "product_name", "category")
        .agg(
            F.sum("quantity").alias("total_units"),
            F.countDistinct("order_id").alias("orders_count")
        )
        .orderBy("gender", F.col("total_units").desc())
    )
else:
    print("One of the required columns in customer order items is all NULL or zero; skipping gender product preference analysis.")

In [165]:
gender_product_pref.show(5)

+------+----------+--------------------+--------------------+-----------+------------+
|gender|product_id|        product_name|            category|total_units|orders_count|
+------+----------+--------------------+--------------------+-----------+------------+
|Female|      1012|       Amazon Tablet|       Winter Sports|       1032|          10|
|Female|      1014|Apple Smartwatch ...|            Dressers|        876|          15|
|Female|      1007|Microsoft Speaker...|            Consoles|        722|          10|
|Female|      1300|                NULL|                NULL|        583|           1|
|Female|      1006|            JBL Case|Computer Accessories|        565|          13|
+------+----------+--------------------+--------------------+-----------+------------+
only showing top 5 rows



New vs returning customers

In [166]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "is_repeat_customer"):
    dataframes["agg_customers"] = dataframes["agg_customers"].withColumn(
        "customer_type",
        F.when(F.col("is_repeat_customer") == 1, F.lit("returning"))
        .otherwise(F.lit("new"))
    )
else:
    print("is_repeat_customer column is all NULL or zero; skipping customer type classification.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_type") and not is_column_all_null_or_zero(dataframes["agg_customers"], "country"):
    new_vs_returning_country = (
        dataframes["agg_customers"]
        .groupBy("country", "customer_type")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("country", "customer_type")
    )
else:
    print("Country or customer_type column is all NULL or zero; skipping new vs returning country analysis.")
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_type") and not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province") and not is_column_all_null_or_zero(dataframes["agg_customers"], "city"):
    new_vs_returning_city = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "city", "customer_type")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("country", "state_province", "city", "customer_type")
    )
else:
    print("One of the required columns (customer_type, country, state_province, city) is all NULL or zero; skipping new vs returning city analysis.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_type") and not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province"):
    new_vs_returning_state = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "customer_type")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country", "state_province", "customer_type")
)
else:
    print("One of the required columns (customer_type, country, state_province) is all NULL or zero; skipping new vs returning state analysis.")

Total & average engagement per customer

In [167]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_sessions") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_pages_viewed") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_products_viewed"):
    engagement_per_customer = dataframes["agg_customers"].select(
        "customer_id",
        "total_sessions",
        "total_pages_viewed",
        "total_products_viewed"
    )

    engagement_overall = dataframes["agg_customers"].agg(
        F.sum("total_sessions").alias("total_sessions_all_customers"),
        F.avg("total_sessions").alias("avg_sessions_per_customer"),
        F.sum("total_pages_viewed").alias("total_pages_viewed_all_customers"),
        F.avg("total_pages_viewed").alias("avg_pages_viewed_per_customer"),
        F.sum("total_products_viewed").alias("total_products_viewed_all_customers"),
        F.avg("total_products_viewed").alias("avg_products_viewed_per_customer")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping engagement analysis.")

Session-to-order behavior

In [168]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "session_conversion_rate") and not is_column_all_null_or_zero(dataframes["agg_customers"], "cart_abandonment_rate"):
    avg_session_to_order = dataframes["agg_customers"].agg(
        F.avg("session_conversion_rate").alias("avg_session_conversion_rate"),
        F.avg("cart_abandonment_rate").alias("avg_cart_abandonment_rate")
    )
else:
    print("One of the required columns (session_conversion_rate, cart_abandonment_rate) is all NULL or zero; skipping session to order analysis.")

distribution percentage for conversion & abandonment

In [169]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "session_conversion_rate"):
    conv_percentage = dataframes["agg_customers"].withColumn(
        "session_conversion_percentage",
        F.when(F.col("session_conversion_rate") < 0.1, "<10%")
        .when(F.col("session_conversion_rate") < 0.25, "10–25%")
        .when(F.col("session_conversion_rate") < 0.5, "25–50%")
        .when(F.col("session_conversion_rate") < 0.75, "50–75%")
        .otherwise("75%+")
    )
    conv_dist = (
        conv_percentage
        .groupBy("session_conversion_percentage")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("session_conversion_percentage")
    )
else:
    print("session_conversion_rate column is all NULL or zero; skipping session conversion percentage calculation.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "cart_abandonment_rate"):
    abandon_percentage = dataframes["agg_customers"].withColumn(
        "cart_abandonment_percentage",
        F.when(F.col("cart_abandonment_rate") < 0.1, "<10%")
        .when(F.col("cart_abandonment_rate") < 0.25, "10–25%")
        .when(F.col("cart_abandonment_rate") < 0.5, "25–50%")
        .when(F.col("cart_abandonment_rate") < 0.75, "50–75%")
        .otherwise("75%+")
    )

    abandon_dist = (
        abandon_percentage
        .groupBy("cart_abandonment_percentage")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("cart_abandonment_percentage")
    )
else:
    print("cart_abandonment_rate column is all NULL or zero; skipping cart abandonment percentage calculation.")

Basic correlation: tenure vs. spending

In [170]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_tenure_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_total_spent") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value"):
    corr_tenure_spend = dataframes["agg_customers"].select(
        F.corr("customer_tenure_days", "order_total_spent").alias("corr_tenure_order_total_spent"),
        F.corr("customer_tenure_days", "customer_lifetime_value").alias("corr_tenure_clv")
    )
else:
    print("One of the required columns (customer_tenure_days, order_total_spent, customer_lifetime_value) is all NULL or zero; skipping correlation analysis.")
    

Tenure buckets vs. average spending

In [171]:
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_tenure_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_total_spent") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    tenure_buckets_df = dataframes["agg_customers"].withColumn(
        "tenure_bucket",
        F.when(F.col("customer_tenure_days") < 30, "<30d")
         .when(F.col("customer_tenure_days") < 90, "30–89d")
         .when(F.col("customer_tenure_days") < 180, "90–179d")
     .when(F.col("customer_tenure_days") < 365, "180–364d")
     .when(F.col("customer_tenure_days") < 730, "1–2y")
     .otherwise("2y+")
)
    

    tenure_spend_stats = (
        tenure_buckets_df
        .groupBy("tenure_bucket")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("order_total_spent").alias("avg_order_total_spent"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.sum("order_total_spent").alias("total_spent")
        )
        .orderBy("tenure_bucket")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping tenure spend analysis.")

# Overall CLV summary

In [172]:
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    clv_summary = dataframes["agg_customers"].agg(
        F.countDistinct("customer_id").alias("customers"),
        F.avg("customer_lifetime_value").alias("avg_clv"),
        F.expr("percentile_approx(customer_lifetime_value, array(0.25, 0.5, 0.75))").alias("clv_percentiles"),
        F.avg("total_revenue").alias("avg_total_revenue"),
        F.avg("avg_order_value").alias("avg_order_value_overall"),
        F.sum("total_revenue").alias("total_revenue_all_customers")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping CLV summary calculation.")

CLV buckets and their spending patterns

In [173]:

if not is_column_all_null_or_zero(dataframes["agg_customers"],"customer_lifetime_value"):
    clv_buckets_df = dataframes["agg_customers"].withColumn(
        "clv_bucket",
        F.when(F.col("customer_lifetime_value") < 100, "<100")
        .when(F.col("customer_lifetime_value") < 500, "100–499")
        .when(F.col("customer_lifetime_value") < 1000, "500–999")
        .when(F.col("customer_lifetime_value") < 5000, "1000–4999")
        .otherwise("5000+")
    )

    clv_bucket_stats = (
        clv_buckets_df
        .groupBy("clv_bucket")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_total_revenue"),
            F.avg("avg_order_value").alias("avg_order_value"),
            F.sum("total_revenue").alias("total_revenue_bucket")
        )
        .orderBy("clv_bucket")
    )
else:
    print("customer_lifetime_value column is all NULL or zero; skipping CLV bucket statistics calculation.")


Relationship between CLV, total_revenue, and avg_order_value

In [174]:
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value"):
    clv_corr = dataframes["agg_customers"].select(
        F.corr("customer_lifetime_value", "total_revenue").alias("corr_clv_total_revenue"),
        F.corr("customer_lifetime_value", "avg_order_value").alias("corr_clv_avg_order_value")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping CLV correlation analysis.")


High-CLV customer segment (top X%)

In [175]:
def high_clv_customers_simple(customers_df, x_percent):
    cutoff_value = customers_df.approxQuantile("customer_lifetime_value", [1 - x_percent/100.0], 0.01)[0]
    return customers_df.filter(F.col("customer_lifetime_value") >= cutoff_value).agg(
        F.countDistinct("customer_id").alias("high_clv_customers"),
        F.avg("customer_lifetime_value").alias("avg_clv"),
        F.avg("avg_order_value").alias("avg_order_value"),
        F.sum("total_revenue").alias("approx_total_revenue")
    )

try:
    if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue"):
        high_clv_stats = high_clv_customers_simple(dataframes["agg_customers"], 10) 
        high_clv_stats.show()
except Exception as e:
    print(f"Error: {e}")

+------------------+-----------------+-----------------+--------------------+
|high_clv_customers|          avg_clv|  avg_order_value|approx_total_revenue|
+------------------+-----------------+-----------------+--------------------+
|                85|5240.015764705882|1302.903323529412|           445401.34|
+------------------+-----------------+-----------------+--------------------+



Top customers in terms of spending 

In [ ]:
top20_customers_by_revenue = dataframes["agg_customers"].select("customer_id", "customer_lifetime_value", "avg_order_value", "total_revenue").orderBy(F.col("total_revenue").desc()).limit(20)


Revenue based on customersegemtns and geolocation

In [ ]:
def revenue_by_segment(df, group_cols):
    grouped = (
        df.groupBy(*group_cols)
          .agg(
              F.countDistinct("customer_id").alias("customer_count"),
              F.sum("total_revenue").alias("segment_revenue")
          )
    )

    total_rev = grouped.agg(F.sum("segment_revenue").alias("total_revenue_all")).first()[0] or 0.0

    result = (
        grouped
        .withColumn(
            "revenue_per_customer",
            F.when(F.col("customer_count") > 0,
                   F.col("segment_revenue") / F.col("customer_count"))
             .otherwise(F.lit(0.0))
        )
        .withColumn(
            "revenue_share",
            F.when(F.lit(total_rev) > 0,
                   F.col("segment_revenue") / F.lit(total_rev))
             .otherwise(F.lit(0.0))
        )
        .orderBy(F.col("segment_revenue").desc_nulls_last())
    )
    return result

if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue"):
    rev_by_country_city = revenue_by_segment(dataframes["agg_customers"], ["country", "city"])
    rev_by_customer_segment = revenue_by_segment(dataframes["agg_customers"], ["customer_segment"])
    rev_by_rfm_segment = revenue_by_segment(dataframes["agg_customers"], ["rfm_segment"])
    rev_by_segment_label = revenue_by_segment(dataframes["agg_customers"], ["customer_segment_label"])
    rev_by_referrer = revenue_by_segment(dataframes["agg_customers"], ["preferred_referrer_source"])
    rev_by_device = revenue_by_segment(dataframes["agg_customers"], ["preferred_device_type"])
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping revenue by segment analysis.")

Discount based analysis, customers contribution based on level of discount 

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_discount_received") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_discount_per_order") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_orders"):
    disc_df = (
        dataframes["agg_customers"]
        .fillna({
            "total_discount_received": 0.0,
            "total_revenue": 0.0,
            "avg_discount_per_order": 0.0,
            "customer_lifetime_value": 0.0,
            "total_orders": 0
        })
        .withColumn(
            "discount_share_of_revenue",
            F.when(F.col("total_revenue") > 0,
                F.col("total_discount_received") / F.col("total_revenue"))
            .otherwise(F.lit(0.0))
        )
    )
    bargain_hunters = (
        disc_df
        .withColumn(
            "is_bargain_hunter",
            F.when(
                (F.col("discount_share_of_revenue") >= 0.3) &
                (F.col("total_orders") >= 3),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
    )
    bargain_summary = (
        bargain_hunters
        .groupBy("is_bargain_hunter")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("discount_share_of_revenue").alias("avg_discount_share"),
            F.avg("avg_discount_per_order").alias("avg_discount_per_order"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_revenue")
        )
    )

    correlation_disc_vs_clv = disc_df.select(
    F.corr("discount_share_of_revenue", "customer_lifetime_value").alias("corr_discount_share_clv"),
    F.corr("avg_discount_per_order", "customer_lifetime_value").alias("corr_avg_discount_clv")
    )

else:
    print("analysis skipped because one of the above columns is missing")

Discount/CLV buckets to see patterns

Bucket by discount_share_of_revenue

In [ ]:
if disc_df:
    disc_bucketed = disc_df.withColumn(
        "discount_intensity_bucket",
        F.when(F.col("discount_share_of_revenue") < 0.1, "<10%")
        .when(F.col("discount_share_of_revenue") < 0.25, "10–24%")
        .when(F.col("discount_share_of_revenue") < 0.5, "25–49%")
        .otherwise("50%+")
    )

    discount_vs_clv = (
        disc_bucketed
        .groupBy("discount_intensity_bucket")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("discount_share_of_revenue").alias("avg_discount_share"),
            F.avg("avg_discount_per_order").alias("avg_discount_per_order"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_revenue")
        )
        .orderBy("discount_intensity_bucket")
    )

else:
    print("skipping discount share analysis because disc_df is null")

Discount Behavior & Margin Pressure

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_discount_received") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_discount_per_order") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    discount_behavior = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "total_revenue",
            "total_discount_received",
            (F.col("total_discount_received") / F.col("total_revenue")).alias("discount_to_revenue_ratio"),
            "avg_discount_per_order",
            "customer_lifetime_value"
        )
    )

    high_discount_customers = (
        discount_behavior
        .filter(F.col("discount_to_revenue_ratio") > 0.3)  # threshold example
        .orderBy(F.col("discount_to_revenue_ratio").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping discount behavior analysis.")

Customer based on avg spent on orders 

In [ ]:
top_customers_by_avg_order = dataframes["agg_customers"].select("customer_id", "customer_lifetime_value", "avg_order_value","avg_items_per_order","total_revenue").orderBy(F.col("avg_order_value").desc())


NameError: name 'dataframes' is not defined

 Cart & Checkout Health (Abandonment & Lost Value)

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_carts_created") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_abandoned_carts") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_purchased_carts") and not is_column_all_null_or_zero(dataframes["agg_customers"], "cart_abandonment_rate") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_abandoned_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_time_in_cart_days"):
    cart_summary = (
        dataframes["agg_customers"]
        .agg(
            F.sum("total_carts_created").alias("total_carts_created"),
            F.sum("total_abandoned_carts").alias("total_abandoned_carts"),
            F.sum("total_purchased_carts").alias("total_purchased_carts"),
            F.avg("cart_abandonment_rate").alias("avg_cart_abandonment_rate"),
            F.sum("total_abandoned_value").alias("total_abandoned_value"),
            F.avg("avg_time_in_cart_days").alias("avg_time_in_cart_days")
        )
    )

    high_value_abandoners = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "total_abandoned_carts",
            "total_abandoned_value",
            "cart_abandonment_rate",
            "total_revenue",
            "customer_lifetime_value"
        )
        .filter(F.col("total_abandoned_value") > 0)
        .orderBy(F.col("total_abandoned_value").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping cart behavior analysis.")

Churn Risk Distribution (Portfolio Health)

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_recency_days"):
    churn_risk_distribution = (
        dataframes["agg_customers"]
        .groupBy("churn_risk")
        .agg(
            F.countDistinct("customer_id").alias("num_customers"),
            F.sum("total_revenue").alias("total_revenue"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("order_recency_days").alias("avg_recency_days")
        )
        .orderBy("churn_risk")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping churn risk analysis.")

High‑CLV Customers at Risk of Churn (Immediate Action List)

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_activity_score") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_recency_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "days_since_last_purchase") and not is_column_all_null_or_zero(dataframes["agg_customers"], "days_since_last_login") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_segment_label") and not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_category"):
    high_clv_at_risk = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "customer_lifetime_value",
            "churn_risk",
            "customer_activity_score",
            "order_recency_days",
            "days_since_last_purchase",
            "days_since_last_login",
            "customer_segment_label",
            "rfm_segment",
            "rfm_category"
        )
        .filter(
            (F.col("churn_risk").isin("medium", "high")) &
            (F.col("customer_lifetime_value") > 0)
        )
        .orderBy(F.col("customer_lifetime_value").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping high CLV at risk analysis.")

RFM Segment Summary (Champions, At Risk, etc.)

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_recency_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_orders") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value"):
    rfm_segment_summary = (
        dataframes["agg_customers"]
        .groupBy("rfm_segment")
        .agg(
            F.countDistinct("customer_id").alias("num_customers"),
            F.sum("total_revenue").alias("total_revenue"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("order_recency_days").alias("avg_recency_days"),
            F.avg("total_orders").alias("avg_total_orders"),
            F.avg("avg_order_value").alias("avg_aov")
        )
        .orderBy(F.col("total_revenue").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping RFM segment summary analysis.")

High‑Intent Non‑Buyers (Fix Funnel / UX)

In [ ]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    high_intent_non_buyers = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "total_products_viewed",
        "wishlist_items_count",
        "total_carts_created",
        "total_purchased_carts",
        "cart_abandonment_rate",
        "session_conversion_rate"
    )
    .filter(
        (F.col("total_revenue") == 0) &
        (
            (F.col("total_products_viewed") > 10) |
            (F.col("wishlist_items_count") > 5) |
            (F.col("total_carts_created") > 3)
        )
    )
    .orderBy(F.col("total_products_viewed").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping high intent non-buyers analysis.")

# Product/category viewing patterns

Category-level viewing effectiveness

In [176]:
if not is_column_all_null_or_zero(dataframes["agg_products"], "category") and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_orders") and not is_column_all_null_or_zero(dataframes["agg_products"], "view_to_purchase_rate") and not is_column_all_null_or_zero(dataframes["agg_products"], "revenue_per_view") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_revenue"):
    category_view_patterns = (
        dataframes["agg_products"]
        .groupBy("category")
        .agg(
            F.countDistinct("product_id").alias("products_in_category"),
            F.sum("total_units_sold").alias("total_units_sold"),
            F.sum("total_orders").alias("total_orders"),
            F.avg("view_to_purchase_rate").alias("avg_view_to_purchase_rate"),
            F.avg("revenue_per_view").alias("avg_revenue_per_view"),
            F.sum("total_revenue").alias("total_revenue")
        )
        .orderBy(F.col("total_revenue").desc_nulls_last())
    )
else:
    print("One of the required columns in agg_products is all NULL or zero; skipping category view patterns analysis.")

One of the required columns in agg_products is all NULL or zero; skipping category view patterns analysis.


Product-level Top-View-to-Purchase Rates

In [177]:
if not is_column_all_null_or_zero(dataframes["agg_products"], "view_to_purchase_rate") and not is_column_all_null_or_zero(dataframes["agg_products"], "revenue_per_view") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_orders"):
    top_view_to_purchase_products = (
        dataframes["agg_products"]
        .select(
            "product_id",
            "product_name",
            "category",
            "view_to_purchase_rate",
            "revenue_per_view",
            "total_units_sold",
            "total_orders"
        )
        .orderBy(F.col("view_to_purchase_rate").desc_nulls_last())
    )
else:
    print("One of the required columns in agg_products is all NULL or zero; skipping top view to purchase products analysis.")

One of the required columns in agg_products is all NULL or zero; skipping top view to purchase products analysis.


# Wishlist usage and conversion rate

Overall wishlist usage and conversion

In [178]:
if not is_column_all_null_or_zero(dataframes["agg_wishlist"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "purchased_date") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "customer_id"):
    wishlist_overall = dataframes["agg_wishlist"].agg(
        F.count("*").alias("total_wishlist_items"),
        F.countDistinct("customer_id").alias("customers_using_wishlist"),
        F.countDistinct("product_id").alias("products_in_wishlist"),
        F.sum(F.when(F.col("purchased_date").isNotNull(), 1).otherwise(0)).alias("wishlist_purchased_items")
    ).withColumn(
        "wishlist_conversion_rate",
        F.col("wishlist_purchased_items") / F.col("total_wishlist_items")
    )
else:
    print("One of the required columns in agg_wishlist is all NULL or zero; skipping wishlist overall analysis.")

Wishlist usage & conversion by product

In [179]:
if not is_column_all_null_or_zero(dataframes["agg_wishlist"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "purchased_date"):
    wishlist_by_product = (
        dataframes["agg_wishlist"]  
        .groupBy("product_id")
        .agg(
            F.count("*").alias("wishlist_adds"),
            F.sum(F.when(F.col("purchased_date").isNotNull(), 1).otherwise(0)).alias("wishlist_purchases")
        )
        .withColumn(
            "wishlist_conversion_rate",
            F.col("wishlist_purchases") / F.col("wishlist_adds")
        )
    )
else:
    print("One of the required columns in agg_wishlist is all NULL or zero; skipping wishlist by product analysis.")

In [180]:
wishlist_by_product.show(3)

+----------+-------------+------------------+------------------------+
|product_id|wishlist_adds|wishlist_purchases|wishlist_conversion_rate|
+----------+-------------+------------------+------------------------+
|      1572|            3|                 3|                     1.0|
|      1159|            2|                 2|                     1.0|
|      1436|            2|                 2|                     1.0|
+----------+-------------+------------------+------------------------+
only showing top 3 rows



Wishlist usage & conversion by customer

In [181]:
if not is_column_all_null_or_zero(dataframes["agg_wishlist"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "purchased_date"):
    wishlist_by_customer = (
        dataframes["agg_wishlist"] 
        .groupBy("customer_id")
        .agg(
            F.count("*").alias("wishlist_adds"),
            F.sum(F.when(F.col("purchased_date").isNotNull(), 1).otherwise(0)).alias("wishlist_purchases")
        )
        .withColumn(
            "wishlist_conversion_rate",
            F.when(F.col("wishlist_adds") > 0,
                F.col("wishlist_purchases") / F.col("wishlist_adds"))
            .otherwise(F.lit(0.0))
        )
    )
else:
    print("customer_id or purchased_date column is all NULL or zero; skipping wishlist by customer analysis.")

Cart creation, abandonment, and recovery statistics

Basic cart creation & status distribution

In [182]:
if not is_column_all_null_or_zero(dataframes["agg_shopping_cart"], "cart_id") and not is_column_all_null_or_zero(dataframes["agg_shopping_cart"], "cart_status"):
    cart_stats_overall = dataframes["agg_shopping_cart"].agg(
        F.countDistinct("cart_id").alias("total_carts"),
        F.count("*").alias("total_cart_lines")
    )
else:
    print("cart_id or cart_status column is all NULL or zero; skipping overall cart statistics analysis.")
if not is_column_all_null_or_zero(dataframes["agg_shopping_cart"], "cart_status"):
    cart_status_dist = (
        dataframes["agg_shopping_cart"]
        .groupBy("cart_status")
        .agg(
            F.countDistinct("cart_id").alias("carts_count"),
        F.count("*").alias("cart_lines_count")
    )
    .orderBy("cart_status")
)
else:
    print("cart_status column is all NULL or zero; skipping cart status distribution analysis.")

Abandonment & recovery (using agg_cart_abandonment_analysis)

In [183]:
dataframes["agg_cart_abandonment_analysis"].show(5)

+-------+-----------+---------------+-----------+----------------+-----------------+-----------------+------------------+------------------------+------------------+-------------------+-----------+-----------------------+----------------+---------------+----------+-------------------+-----------------------+---------------+------------------+----------------------+
|cart_id|cart_status|cart_added_date|customer_id|cart_items_count|session_converted|time_in_cart_days|time_in_cart_hours|recovery_potential_score|  cart_total_value|cart_avg_item_price|device_used|abandoned_cart_category|first_added_date|last_added_date|session_id|cart_status_derived|cart_abandonment_reason|cart_value_tier|cart_size_category|abandonment_risk_score|
+-------+-----------+---------------+-----------+----------------+-----------------+-----------------+------------------+------------------------+------------------+-------------------+-----------+-----------------------+----------------+---------------+----------

In [184]:
if not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_id") and not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_status"):
    cart_abandon_overall = dataframes["agg_cart_abandonment_analysis"].agg(
        F.countDistinct("cart_id").alias("total_carts_tracked"),
        F.countDistinct(F.when(F.col("cart_status") == "Abandoned", F.col("cart_id"))).alias("abandoned_carts"),
        F.countDistinct(F.when(F.col("cart_status") == "Converted", F.col("cart_id"))).alias("converted_carts")
    ).withColumn(
        "abandonment_rate",
        F.col("abandoned_carts") / F.col("total_carts_tracked")
    ).withColumn(
        "purchase_rate",
        F.col("converted_carts") / F.col("total_carts_tracked")
    )
else:
    print("cart_id or cart_status column is all NULL or zero; skipping cart abandonment overall analysis.")

In [185]:
cart_abandon_overall.show()

+-------------------+---------------+---------------+-----------------+------------------+
|total_carts_tracked|abandoned_carts|converted_carts| abandonment_rate|     purchase_rate|
+-------------------+---------------+---------------+-----------------+------------------+
|               1585|           1003|            441|0.632807570977918|0.2782334384858044|
+-------------------+---------------+---------------+-----------------+------------------+



Value and size characteristics of abandoned vs purchased carts

In [186]:
if not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_status"):
    cart_value_stats = (
        dataframes["agg_cart_abandonment_analysis"]
        .groupBy("cart_status")
        .agg(
            F.countDistinct("cart_id").alias("carts_count"),
            F.avg("cart_total_value").alias("avg_cart_value"),
        F.avg("cart_items_count").alias("avg_cart_items"),
        F.avg("time_in_cart_days").alias("avg_time_in_cart_days"),
        F.avg("recovery_potential_score").alias("avg_recovery_potential_score")
    )
    .orderBy("cart_status")
)
else:
    print("cart_status column is all NULL or zero; skipping cart value statistics analysis.")

Recovery opportunity: high-value abandoned carts

In [187]:
if not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_status") and not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_total_value"):
    high_value_abandoned = (
        dataframes["agg_cart_abandonment_analysis"]
        .filter(
            (F.col("cart_status") == "abandoned") &
            (F.col("cart_total_value") >= 100)  # threshold – adjust as needed
        )
        .select(
            "cart_id",
            "customer_id",
            "cart_total_value",
            "cart_items_count",
            "time_in_cart_days",
            "recovery_potential_score",
            "abandonment_risk_score"
        )
    )
else:
    print("cart_status or cart_total_value column is all NULL or zero; skipping high-value abandoned carts analysis.")